In [ ]:
"""
Smart MCQ Solver — DeBERTa-v3 Multiple-Choice fine-tuning pipeline
====================================================================

Data format expected:
    train.csv : id, prompt, A, B, C, D, E, answer
    test.csv  : id, prompt, A, B, C, D, E

Output:
    submission.csv : id, prediction   (prediction = "B A D" style, top-3 space-separated letters)

Install (Colab):
    !pip install -q transformers datasets accelerate scikit-learn torch wandb

Run:
    python train_mcq_solver.py
"""

In [1]:
!pip install -q transformers datasets accelerate scikit-learn torch wandb

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

In [4]:

# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "microsoft/deberta-v3-base"   # swap to "-large" if you have the GPU budget
TRAIN_CSV = "/content/train.csv"
TEST_CSV = "/content/test.csv"
OUTPUT_DIR = "./mcq_model"
MAX_LEN = 256
OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {c: i for i, c in enumerate(OPTIONS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

USE_WANDB = False  # flip to True and set project name if you want W&B tracking
if USE_WANDB:
    import wandb
    wandb.init(project="mcq-science-exam", name="deberta-v3-base-mc")


In [5]:

# -----------------------------
# Load & prep data
# -----------------------------
def load_data():
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df["label"] = train_df["answer"].map(LABEL2ID)
    return train_df, test_df


def to_hf_dataset(df, has_label=True):
    cols = ["id", "prompt"] + OPTIONS + (["label"] if has_label else [])
    return Dataset.from_pandas(df[cols].reset_index(drop=True))

In [6]:

# -----------------------------
# Tokenization: build 5 (prompt, option) pairs per row
# -----------------------------
def preprocess(examples, tokenizer):
    first_sentences = [[p] * 5 for p in examples["prompt"]]
    second_sentences = [
        [examples[opt][i] for opt in OPTIONS] for i in range(len(examples["prompt"]))
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    # reshape flat list back into [batch, 5, seq_len]
    out = {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
    return out

In [7]:

# -----------------------------
# Data collator for multiple choice (dynamic padding per-batch)
# -----------------------------
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else None
        labels = [feature.pop(label_name) for feature in features] if label_name else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch



In [8]:
# -----------------------------
# MAP@3 metric
# -----------------------------
def map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for pred_row, true_label in zip(top3, labels):
        if true_label == pred_row[0]:
            scores.append(1.0)
        elif true_label == pred_row[1]:
            scores.append(1.0 / 2)
        elif true_label == pred_row[2]:
            scores.append(1.0 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "map@3": map_at_3(logits, labels)}


In [13]:

# -----------------------------
# Main
# -----------------------------
def main():
    train_df, test_df = load_data()

    # stratified split for a held-out validation set
    tr_df, val_df = train_test_split(
        train_df, test_size=0.15, random_state=42, stratify=train_df["label"]
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    train_ds = to_hf_dataset(tr_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    val_ds = to_hf_dataset(val_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    test_ds = to_hf_dataset(test_df, has_label=False).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )

    collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map@3",
        learning_rate=1e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=4,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=20,
        report_to=["wandb"] if USE_WANDB else [],
        bf16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print("Validation results:", trainer.evaluate())

    # -----------------------------
    # Predict on test set -> top-3 ranked letters per row
    # -----------------------------
    preds = trainer.predict(test_ds)
    logits = preds.predictions  # shape [n_test, 5]
    top3_idx = np.argsort(-logits, axis=1)[:, :3]
    top3_letters = [" ".join(ID2LABEL[i] for i in row) for row in top3_idx]

    submission = pd.DataFrame({"id": test_df["id"], "prediction": top3_letters})
    submission.to_csv("/content/drive/MyDrive/submission.csv", index=False)
    print(submission.head())
    print("Saved submission to /content/drive/MyDrive/submission.csv")

    if USE_WANDB:
        wandb.finish()


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weigh

Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Map@3
6.449512,1.609375,4,0.183333,0.381667


Validation results: {'eval_loss': 1.609375, 'eval_accuracy': 0.18333333333333332, 'eval_map@3': 0.38166666666666665}


   id prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C
Saved submission to /content/drive/MyDrive/submission.csv


In [6]:

import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from peft import LoraConfig, get_peft_model, TaskType
from transformers import set_seed
from sklearn.model_selection import StratifiedKFold


# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "microsoft/deberta-v3-large"   # bigger backbone, LoRA keeps it trainable
TRAIN_CSV = "/content/train.csv"
TEST_CSV = "/content/test.csv"
OUTPUT_DIR = "./mcq_lora_model"
MAX_LEN = 256
OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {c: i for i, c in enumerate(OPTIONS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

# target_modules differ by architecture family:
#   DeBERTa-v3 -> "query_proj", "value_proj"
#   BERT/RoBERTa -> "query", "value"
LORA_TARGET_MODULES = ["query_proj", "value_proj"] if "deberta" in MODEL_NAME else ["query", "value"]

USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.init(project="mcq-science-exam", name="deberta-v3-large-lora")



In [7]:

# -----------------------------
# Load & prep data
# -----------------------------
def load_data():
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df["label"] = train_df["answer"].map(LABEL2ID)
    return train_df, test_df


def to_hf_dataset(df, has_label=True):
    cols = ["id", "prompt"] + OPTIONS + (["label"] if has_label else [])
    return Dataset.from_pandas(df[cols].reset_index(drop=True))


def preprocess(examples, tokenizer):
    first_sentences = [[p] * 5 for p in examples["prompt"]]
    second_sentences = [
        [examples[opt][i] for opt in OPTIONS] for i in range(len(examples["prompt"]))
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}


@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else None
        labels = [feature.pop(label_name) for feature in features] if label_name else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


def map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for pred_row, true_label in zip(top3, labels):
        if true_label == pred_row[0]:
            scores.append(1.0)
        elif true_label == pred_row[1]:
            scores.append(0.5)
        elif true_label == pred_row[2]:
            scores.append(1.0 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "map@3": map_at_3(logits, labels)}


In [9]:
def main():
    train_df, test_df = load_data()
    tr_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    set_seed(42)
    base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    # ---- Wrap with LoRA ----
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,   # closest available task type; MC head behaves like a scoring head
        r=16,
        lora_alpha=132,
        lora_dropout=0.1,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=["classifier", "pooler"],
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()# sanity check: should show a tiny % of total params


    train_ds = to_hf_dataset(tr_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    val_ds = to_hf_dataset(val_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    test_ds = to_hf_dataset(test_df, has_label=False).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map@3",
        learning_rate=5e-5,              # LoRA typically wants a higher LR than full fine-tune
        per_device_train_batch_size=2,   # smaller since backbone is larger
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=5,              # LoRA often needs a few more epochs to converge
        weight_decay=0.05,
        warmup_ratio=0.1,
        logging_steps=20,
        save_total_limit=1,
        report_to=["wandb"] if USE_WANDB else [],
        fp16=False,
        bf16=False,                      # keep mixed precision off, same instability risk as full fine-tune
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print("Validation results:", trainer.evaluate())

    preds = trainer.predict(test_ds)
    logits = preds.predictions
    top3_idx = np.argsort(-logits, axis=1)[:, :3]
    top3_letters = [" ".join(ID2LABEL[i] for i in row) for row in top3_idx]

    submission = pd.DataFrame({"id": test_df["id"], "prediction": top3_letters})
    submission.to_csv("/content/drive/MyDrive/submission_lora_2.csv", index=False)
    print(submission.head())
    print("Saved submission to /content/drive/MyDrive/submission_lora_2.csv")

    # save just the LoRA adapter (small, a few MB) rather than the full backbone
    model.save_pretrained("./mcq_lora_adapter")

    if USE_WANDB:
        wandb.finish()


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weig

trainable params: 2,623,489 || all params: 437,686,274 || trainable%: 0.5994


Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,12.315674,1.324219,0.626667,0.757222
2,5.606481,0.317627,0.900000,0.946111
3,2.823729,0.163818,0.956667,0.977778
4,2.397058,0.106079,0.973333,0.986667
5,2.098731,0.103088,0.986667,0.993333


Training Loss,Validation Loss,Epoch,Accuracy,Map@3
2.098731,0.103088,5,0.986667,0.993333


Validation results: {'eval_loss': 0.10308837890625, 'eval_accuracy': 0.9866666666666667, 'eval_map@3': 0.9933333333333333}


   id prediction
0   1      A C D
1   2      B A C
2   3      B E D
3   4      E C A
4   5      C D A
Saved submission to /content/drive/MyDrive/submission_lora_1.csv


In [4]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
